# 💸 Personal Finance Spending Clusters

## What This Project Does
In this project we will use **unsupervised machine learning** to group monthly spending patterns into clusters. Unlike the other projects in this series, we have no labels to predict — instead, the algorithm will discover hidden structure in the data on its own.

We will use the **Personal Finance** dataset, aggregate spending by category per month, and apply **K-Means clustering** to automatically group months with similar spending behaviour. This can reveal patterns like "high entertainment month", "savings mode month", or "holiday season month".

## What Is Unsupervised Learning?
In **supervised learning** (Projects 1–4), we had labeled data: we knew the correct answer for each data point (positive/negative, spam/ham, good/bad wine). The model learned a mapping from inputs to those known labels.

In **unsupervised learning**, we have **no labels**. There is no "correct" answer. Instead, we ask: *"Are there natural groupings or patterns in this data?"* K-Means finds groups (clusters) of data points that are similar to each other. The algorithm doesn't know what the clusters represent — that's our job to interpret.

## What You Will Learn
- The difference between **supervised and unsupervised learning**
- How to **aggregate raw transaction data** into a feature matrix for ML
- How the **Elbow Method** helps pick the right number of clusters
- How **K-Means clustering** works and what it outputs
- How to use **PCA** to visualize high-dimensional clusters in 2D
- How to interpret cluster results with **interactive Plotly charts**

## Dataset
- **Source:** [Personal Finance on Kaggle](https://www.kaggle.com/datasets/bukolafatunde/personal-finance)
- **File to upload:** `personal_finance.csv`

> 💡 **Tip:** Download the dataset from Kaggle before running this notebook.

In [ ]:
# ── STEP 1: Install Extra Libraries ──────────────────────────────────────────
# Plotly gives us interactive charts. All other libraries are pre-installed in Colab.

!pip install -q plotly

In [ ]:
# ── STEP 1 (continued): Import All Libraries ─────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
print('✅ All libraries imported!')

In [ ]:
# ── STEP 2: Upload the Dataset ────────────────────────────────────────────────
# Click 'Choose Files' and upload personal_finance.csv.

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Uploaded: {filename}')

In [ ]:
# ── STEP 2 (continued): Load and Inspect the Dataset ─────────────────────────

df = pd.read_csv(filename)

print(f'Shape: {df.shape}')
print(f'\nColumn names: {list(df.columns)}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# ── STEP 3: Exploratory Data Analysis ────────────────────────────────────────
# Before clustering, we explore the data to understand:
#   • Which spending categories are most/least expensive on average
#   • How spending varies over time
#
# First, standardize column names to lowercase for easier coding.
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

# Auto-detect column names: look for date, category, and amount columns
# (The dataset may use slightly different names depending on the version)
date_col     = [c for c in df.columns if 'date' in c][0]
category_col = [c for c in df.columns if 'categor' in c][0]
amount_col   = [c for c in df.columns if 'amount' in c or 'spend' in c or 'value' in c][0]

print(f'Date column     : {date_col}')
print(f'Category column : {category_col}')
print(f'Amount column   : {amount_col}')

# Parse dates and extract month/year
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df['year_month'] = df[date_col].dt.to_period('M')   # Groups by month (e.g. '2021-03')

# Make sure amounts are positive numbers (some datasets store expenses as negatives)
df[amount_col] = df[amount_col].abs()

In [ ]:
# ── STEP 3 (continued): Total Spending by Category ────────────────────────────
# Sum all spending for each category across the entire dataset.
# This gives us a high-level view of where money is going.

category_totals = (
    df.groupby(category_col)[amount_col]
    .sum()
    .sort_values(ascending=True)
)

plt.figure(figsize=(9, 5))
category_totals.plot(kind='barh', color='steelblue')
plt.title('Total Spending by Category')
plt.xlabel('Total Amount Spent')
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 3 (continued): Monthly Spending Trend ────────────────────────────────
# Aggregate total spending per month to see if spending fluctuates seasonally.
# We use Plotly for an interactive chart — hover to see exact values.

monthly_totals = (
    df.groupby('year_month')[amount_col]
    .sum()
    .reset_index()
)
monthly_totals['year_month'] = monthly_totals['year_month'].astype(str)   # Convert Period to string for Plotly

fig = px.line(
    monthly_totals,
    x='year_month',
    y=amount_col,
    markers=True,
    title='Monthly Total Spending Over Time',
    labels={amount_col: 'Total Spent', 'year_month': 'Month'},
    template='plotly_white'
)
fig.show()

In [ ]:
# ── STEP 4: Feature Engineering — Create the Feature Matrix ──────────────────
# For K-Means clustering, we need a feature matrix where:
#   • Each ROW represents ONE month
#   • Each COLUMN represents spending in ONE category for that month
#
# This is called a 'pivot table'. For example:
#   Month   | Food | Transport | Entertainment | ...
#   2021-01 | 350  | 80        | 120           | ...
#   2021-02 | 290  | 95        | 200           | ...
#
# We use .pivot_table() with aggfunc='sum' to total spending per category per month.
# fill_value=0 fills months where a category had no spending with 0.

feature_matrix = df.pivot_table(
    index='year_month',
    columns=category_col,
    values=amount_col,
    aggfunc='sum',
    fill_value=0
)

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'(rows = months, columns = spending categories)')
print(f'\nCategories: {list(feature_matrix.columns)}')
feature_matrix.head()

In [ ]:
# ── STEP 5: Scale Features with StandardScaler ────────────────────────────────
# K-Means assigns each data point to the nearest cluster centre based on
# EUCLIDEAN DISTANCE (straight-line distance in the feature space).
#
# Why does K-Means need scaling?
# Imagine 'Housing' costs $1,200/month and 'Coffee' costs $25/month.
# Without scaling, the distance between two months is dominated by Housing
# and Coffee barely matters. K-Means would cluster based almost entirely on
# rent differences and ignore all the other categories.
# StandardScaler transforms each feature to mean=0, std=1 so every category
# contributes equally to the distance calculation.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_matrix)

print(f'Scaled feature matrix shape: {X_scaled.shape}')

In [ ]:
# ── STEP 6: Elbow Method — How Many Clusters? ────────────────────────────────
# K-Means requires us to specify K (the number of clusters) in advance.
# But how do we know what K to choose? We use the Elbow Method:
#
# For each value of K (1, 2, 3, ... 10), we:
#   1. Train K-Means with that K
#   2. Record the 'inertia' — the sum of squared distances from each point to
#      its nearest cluster centre (lower = tighter, better-formed clusters)
#
# As K increases, inertia always decreases (more clusters = points are closer
# to their centres). We look for the 'elbow' — the point where adding more
# clusters stops significantly reducing inertia. That K is the sweet spot.
#
# Think of it like: fitting one line through a cloud of points has high error;
# fitting 5 lines reduces error a lot; fitting 100 lines barely helps more.
# The 'elbow' is where the benefit of adding more lines plateaus.

inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', markersize=8)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Sum of Squared Distances)')
plt.title('Elbow Method — Finding the Optimal Number of Clusters')
plt.xticks(K_range)
plt.grid(True)
plt.tight_layout()
plt.show()
print('Look for the "elbow" — the K value where the curve starts to flatten out.')
print('That is your ideal number of clusters.')

In [ ]:
# ── STEP 7: Train K-Means with the Chosen K ───────────────────────────────────
# Based on the elbow chart above, choose the K value where the curve starts to
# flatten. Typical values for monthly spending data are K=3 or K=4.
# You can change CHOSEN_K below after looking at your elbow chart.
#
# n_init=10 means K-Means runs 10 times with different random starting points
# and keeps the best result (K-Means can converge to different local minima
# depending on starting positions).

CHOSEN_K = 3   # ← Change this based on what you see in the elbow chart above

kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=10)
kmeans.fit(X_scaled)

print(f'K-Means trained with K={CHOSEN_K}')
print(f'Final inertia: {kmeans.inertia_:.2f}')

In [ ]:
# ── STEP 8: Add Cluster Labels Back to the DataFrame ─────────────────────────
# After training, kmeans.labels_ contains a cluster number (0, 1, 2, ...) for
# each month. We add these back to our feature matrix so we can analyse what
# each cluster looks like.

feature_matrix['cluster'] = kmeans.labels_

print('Cluster assignments:')
print(feature_matrix['cluster'].value_counts().sort_index())
print(f'\nMonths per cluster (out of {len(feature_matrix)} total months)')

In [ ]:
# ── STEP 9: Analyze Each Cluster ─────────────────────────────────────────────
# For each cluster, compute the AVERAGE spending per category.
# This lets us understand what makes each cluster different — and give it
# a meaningful label like 'high food spender' or 'low overall spender'.

category_cols = [c for c in feature_matrix.columns if c != 'cluster']
cluster_profiles = feature_matrix.groupby('cluster')[category_cols].mean()

print('=== Average Spending Per Category Per Cluster ===')
print(cluster_profiles.round(2).to_string())
print('\nLook at which cluster has the highest values in each column.')
print('Try to label each cluster with a description like:')
print('  Cluster 0: "Frugal month"  |  Cluster 1: "High food & entertainment"  |  Cluster 2: "Big spending month"')

In [ ]:
# ── STEP 10: PCA Scatter Plot — Visualizing Clusters in 2D ───────────────────
# Our feature matrix has one column per spending category, so it exists in a
# high-dimensional space (e.g. 8 dimensions for 8 categories). We cannot
# visualize 8 dimensions directly.
#
# PCA (Principal Component Analysis) solves this by finding the 2 directions
# in the high-dimensional space that capture the MOST VARIANCE (spread) in the
# data, then projecting every data point onto those 2 directions.
#
# Think of it like looking at a 3D object — the shadow you see on the wall
# is a 2D projection that still captures the overall shape. PCA gives us the
# 'most informative shadow' of our high-dimensional data.
#
# After PCA, we can plot each month as a dot on a 2D chart, coloured by cluster.

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)    # Reduce to 2 dimensions

pca_df = pd.DataFrame({
    'PC1': pca_coords[:, 0],
    'PC2': pca_coords[:, 1],
    'Cluster': feature_matrix['cluster'].astype(str),
    'Month': feature_matrix.index.astype(str)
})

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Cluster',
    hover_data=['Month'],
    title='Spending Clusters Visualized with PCA (2D)',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'},
    template='plotly_white',
    size_max=12
)
fig.update_traces(marker=dict(size=10, opacity=0.8))
fig.show()

variance_explained = pca.explained_variance_ratio_.sum() * 100
print(f'\nVariance explained by 2 components: {variance_explained:.1f}%')
print('If clusters are well-separated in the chart, K-Means found distinct groups!')

In [ ]:
# ── STEP 11: Interactive Bar Chart — Average Spending per Cluster ─────────────
# This grouped bar chart lets us compare average spending per category across
# all clusters simultaneously. Hover over each bar to see the exact values.
# This is the key visualization for interpreting what each cluster represents.

# Melt the cluster_profiles into a long-format DataFrame for Plotly
cluster_long = cluster_profiles.reset_index().melt(
    id_vars='cluster',
    var_name='Category',
    value_name='Average Spending'
)
cluster_long['cluster'] = 'Cluster ' + cluster_long['cluster'].astype(str)

fig = px.bar(
    cluster_long,
    x='Category',
    y='Average Spending',
    color='cluster',
    barmode='group',
    title='Average Spending per Category by Cluster',
    template='plotly_white'
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()
print('Each cluster group of bars shows the average spending for that category per cluster.')

## 🎉 Summary: What You Learned

Congratulations on completing the **Personal Finance Spending Clusters** project!

| Step | What You Did | Why It Matters |
|------|--------------|----------------|
| Unsupervised Learning | No labels — let the algorithm find structure | Discover patterns you didn't know to look for |
| Feature Engineering | Pivoted transactions → monthly spending matrix | Transformed raw data into an ML-ready format |
| StandardScaler | Normalized spending amounts | K-Means needs equal-scale features for fair distance computation |
| Elbow Method | Found optimal K | Prevents too few or too many clusters |
| K-Means | Grouped months by spending similarity | Automatically finds spending behaviour types |
| PCA | Reduced dimensions to 2D | Visualizes high-dimensional clusters on a scatter plot |
| Cluster Analysis | Computed average spending per cluster | Interprets what each cluster actually means |

### How You Could Use This on Your Own Bank Data
1. **Export your bank statement** as a CSV (most banks offer this under "Download transactions")
2. **Categorize transactions** — either manually or use a budgeting app like Mint/YNAB to export already-categorized data
3. **Replace the dataset** in this notebook with your CSV and re-run all cells
4. Look at the clusters — do you notice patterns? High-spending months? Seasonal trends?
5. Use the insights to set **monthly budgets** for categories where you overspend

### Next Steps
- Try **DBSCAN** — a different clustering algorithm that doesn't require specifying K in advance
- Try **hierarchical clustering** (dendrogram) to see how months relate to each other at multiple scales
- Add more features like **income, savings rate, or debt payments** to enrich the clustering
- Try **anomaly detection** to automatically flag unusually expensive months